In [12]:
import os
import re
import pandas as pd
from datetime import datetime
from pathlib import Path

def find_project_root(start=Path.cwd(), marker_files=('requirements.txt','LEAF_QUICKSTART.md')):
    p = Path(start).resolve()
    while True:
        if any((p / m).exists() for m in marker_files):
            return p
        if p.parent == p:
            return Path.cwd().resolve()
        p = p.parent

try:
    ROOT_DIR = Path(__file__).resolve().parent.parent.parent
except NameError:
    ROOT_DIR = find_project_root(Path.cwd())


def check_missing_files(folder_path):
    # 1. Thiết lập khoảng thời gian mong muốn theo tháng
    start_month = datetime(2013, 1, 1)
    end_month = datetime(2026, 1, 1)

    # Danh sách các tháng cần có: 01/2013 -> 01/2026
    expected_months = pd.date_range(start=start_month, end=end_month, freq='MS')

    # 2. Lấy danh sách file trong thư mục
    try:
        files = os.listdir(folder_path)
    except FileNotFoundError:
        print("Đường dẫn thư mục không tồn tại!")
        return

    xlsx_files = [f for f in files if f.endswith(".xlsx")]
    print(f"Tổng số file .xlsx trong thư mục: {len(xlsx_files)}")

    # 3. Trích xuất tháng/năm từ tên file
    # File mẫu: GiaoDichNN_VNINDEX_01_1_2013_02_01_2013.xlsx
    # Định dạng: Tháng_Ngày_Năm_Tháng_Ngày_Năm
    pattern_two_dates = re.compile(r"VNINDEX_(\d{1,2})_(\d{1,2})_(\d{4})_(\d{1,2})_(\d{1,2})_(\d{4})")
    pattern_short = re.compile(r"VNINDEX_(\d{1,2})_(\d{4})")

    existing_months = set()

    for filename in xlsx_files:
        match = pattern_two_dates.search(filename)
        if match:
            # groups: month_start, day_start, year_start, month_end, day_end, year_end
            try:
                month_start = int(match.group(1))
                year_start = int(match.group(3))
                if 1 <= month_start <= 12:
                    existing_months.add((year_start, month_start))
                    continue
            except ValueError:
                pass

        match = pattern_short.search(filename)
        if match:
            try:
                month = int(match.group(1))
                year = int(match.group(2))
                if 1 <= month <= 12:
                    existing_months.add((year, month))
                    continue
            except ValueError:
                pass

    # 4. Kiểm tra thiếu sót theo tháng
    missing_months = []
    for m in expected_months:
        key = (m.year, m.month)
        if key not in existing_months:
            missing_months.append(key)

    # 5. Thông báo kết quả
    if not missing_months:
        print(f"Chúc mừng! Bạn đã có đủ file theo tháng từ {start_month.strftime('%m/%Y')} đến {end_month.strftime('%m/%Y')}.")
    else:
        print(f"Phát hiện thiếu {len(missing_months)} tháng.")
        print("Danh sách các tháng bị thiếu (liệt kê tối đa 20 tháng đầu):")
        for y, mo in missing_months[:20]:
            print(f"- Thiếu file cho tháng: {mo:02d}/{y}")
        if len(missing_months) > 20:
            print("...")

    return missing_months

# Thay đường dẫn thư mục của bạn vào đây
folder_path = ROOT_DIR / "data" / "raw" / "VNINDEX"
# missing = check_missing_files(folder_path)


In [13]:
missing = check_missing_files(folder_path)

Tổng số file .xlsx trong thư mục: 156
Phát hiện thiếu 1 tháng.
Danh sách các tháng bị thiếu (liệt kê tối đa 20 tháng đầu):
- Thiếu file cho tháng: 01/2026


In [5]:
def check_corrupt_or_empty_files(folder_path):
    import warnings
    warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
    
    corrupt_files = []
    empty_files = []
    
    try:
        files = os.listdir(folder_path)
    except FileNotFoundError:
        print("Đường dẫn thư mục không tồn tại!")
        return [], []
        
    xlsx_files = [f for f in files if f.endswith(".xlsx")]
    print(f"Đang kiểm tra {len(xlsx_files)} file .xlsx ...")
    
    for filename in xlsx_files:
        filepath = os.path.join(folder_path, filename)
        try:
            # Try to read the excel file
            df = pd.read_excel(filepath)
            
            # Check if dataframe is empty
            if df.empty or len(df) == 0:
                empty_files.append(filename)
                
        except Exception as e:
            # If pandas fails to read it, it's likely corrupt/can't be opened
            corrupt_files.append((filename, str(e)))
            
    print("\n--- KẾT QUẢ KIỂM TRA DỮ LIỆU VÀ FILE ---")
    if not corrupt_files and not empty_files:
        print("Tuyệt vời! Tất cả các file đều mở được và có dữ liệu.")
    
    if corrupt_files:
        print(f"\nPhát hiện {len(corrupt_files)} file không thể mở được (lỗi/corrupt):")
        for f, err in corrupt_files[:30]:
            print(f"- {f}")
        if len(corrupt_files) > 30: print("...")
            
    if empty_files:
        print(f"\nPhát hiện {len(empty_files)} file mở được nhưng KHÔNG có dữ liệu (empty):")
        for f in empty_files[:30]:
            print(f"- {f}")
        if len(empty_files) > 30: print("...")
            
    return corrupt_files, empty_files

corrupt, empty = check_corrupt_or_empty_files(folder_path)

Đang kiểm tra 156 file .xlsx ...

--- KẾT QUẢ KIỂM TRA DỮ LIỆU VÀ FILE ---

Phát hiện 24 file không thể mở được (lỗi/corrupt):
- GiaoDichNN_VNINDEX_01_01_2020_02_01_2020.xlsx
- GiaoDichNN_VNINDEX_01_01_2021_02_01_2021.xlsx
- GiaoDichNN_VNINDEX_02_01_2019_03_01_2019.xlsx
- GiaoDichNN_VNINDEX_02_01_2020_03_01_2020.xlsx
- GiaoDichNN_VNINDEX_03_01_2019_04_01_2019.xlsx
- GiaoDichNN_VNINDEX_03_01_2020_04_01_2020.xlsx
- GiaoDichNN_VNINDEX_03_01_2021_04_01_2021.xlsx
- GiaoDichNN_VNINDEX_04_01_2019_05_01_2019.xlsx
- GiaoDichNN_VNINDEX_04_01_2021_05_01_2021.xlsx
- GiaoDichNN_VNINDEX_05_01_2019_06_01_2019.xlsx
- GiaoDichNN_VNINDEX_05_01_2020_06_01_2020.xlsx
- GiaoDichNN_VNINDEX_05_01_2021_06_01_2021.xlsx
- GiaoDichNN_VNINDEX_06_01_2020_07_01_2020.xlsx
- GiaoDichNN_VNINDEX_06_01_2021_07_01_2021.xlsx
- GiaoDichNN_VNINDEX_07_01_2019_08_01_2019.xlsx
- GiaoDichNN_VNINDEX_07_01_2020_08_01_2020.xlsx
- GiaoDichNN_VNINDEX_07_01_2021_08_01_2021.xlsx
- GiaoDichNN_VNINDEX_08_01_2019_09_01_2019.xlsx
- GiaoDic

### Kiểm tra vn30

In [8]:
import os
import re
import pandas as pd
from datetime import datetime
from pathlib import Path

def find_project_root(start=Path.cwd(), marker_files=('requirements.txt','LEAF_QUICKSTART.md')):
    p = Path(start).resolve()
    while True:
        if any((p / m).exists() for m in marker_files):
            return p
        if p.parent == p:
            return Path.cwd().resolve()
        p = p.parent

try:
    ROOT_DIR = Path(__file__).resolve().parent.parent.parent
except NameError:
    ROOT_DIR = find_project_root(Path.cwd())


def check_missing_files(folder_path):
    # 1. Thiết lập khoảng thời gian mong muốn theo tháng
    start_month = datetime(2013, 1, 1)
    end_month = datetime(2026, 1, 1)

    # Danh sách các tháng cần có: 01/2013 -> 01/2026
    expected_months = pd.date_range(start=start_month, end=end_month, freq='MS')

    # 2. Lấy danh sách file trong thư mục
    try:
        files = os.listdir(folder_path)
    except FileNotFoundError:
        print("Đường dẫn thư mục không tồn tại!")
        return

    xlsx_files = [f for f in files if f.endswith(".xlsx")]
    print(f"Tổng số file .xlsx trong thư mục: {len(xlsx_files)}")

    # 3. Trích xuất tháng/năm từ tên file
    # File mẫu: GiaoDichNN_VNINDEX_01_1_2013_02_01_2013.xlsx
    # Định dạng: Tháng_Ngày_Năm_Tháng_Ngày_Năm
    pattern_two_dates = re.compile(r"ALL_(\d{1,2})_(\d{1,2})_(\d{4})_(\d{1,2})_(\d{1,2})_(\d{4})")
    pattern_short = re.compile(r"ALL_(\d{1,2})_(\d{4})")

    existing_months = set()

    for filename in xlsx_files:
        match = pattern_two_dates.search(filename)
        if match:
            # groups: month_start, day_start, year_start, month_end, day_end, year_end
            try:
                month_start = int(match.group(1))
                year_start = int(match.group(3))
                if 1 <= month_start <= 12:
                    existing_months.add((year_start, month_start))
                    continue
            except ValueError:
                pass

        match = pattern_short.search(filename)
        if match:
            try:
                month = int(match.group(1))
                year = int(match.group(2))
                if 1 <= month <= 12:
                    existing_months.add((year, month))
                    continue
            except ValueError:
                pass

    # 4. Kiểm tra thiếu sót theo tháng
    missing_months = []
    for m in expected_months:
        key = (m.year, m.month)
        if key not in existing_months:
            missing_months.append(key)

    # 5. Thông báo kết quả
    if not missing_months:
        print(f"Chúc mừng! Bạn đã có đủ file theo tháng từ {start_month.strftime('%m/%Y')} đến {end_month.strftime('%m/%Y')}.")
    else:
        print(f"Phát hiện thiếu {len(missing_months)} tháng.")
        print("Danh sách các tháng bị thiếu (liệt kê tối đa 20 tháng đầu):")
        for y, mo in missing_months[:20]:
            print(f"- Thiếu file cho tháng: {mo:02d}/{y}")
        if len(missing_months) > 20:
            print("...")

    return missing_months

# Thay đường dẫn thư mục của bạn vào đây
folder_path = ROOT_DIR / "data" / "raw" / "VN30"
# missing = check_missing_files(folder_path)


In [9]:
missing = check_missing_files(folder_path)

Tổng số file .xlsx trong thư mục: 156
Phát hiện thiếu 1 tháng.
Danh sách các tháng bị thiếu (liệt kê tối đa 20 tháng đầu):
- Thiếu file cho tháng: 01/2026


In [10]:
corrupt, empty = check_corrupt_or_empty_files(folder_path)

Đang kiểm tra 156 file .xlsx ...

--- KẾT QUẢ KIỂM TRA DỮ LIỆU VÀ FILE ---

Phát hiện 20 file không thể mở được (lỗi/corrupt):
- GiaoDichNN_ALL_01_01_2020_02_01_2020.xlsx
- GiaoDichNN_ALL_01_01_2021_02_01_2021.xlsx
- GiaoDichNN_ALL_02_01_2019_03_01_2019.xlsx
- GiaoDichNN_ALL_02_01_2020_03_01_2020.xlsx
- GiaoDichNN_ALL_03_01_2019_04_01_2019.xlsx
- GiaoDichNN_ALL_03_01_2020_04_01_2020.xlsx
- GiaoDichNN_ALL_04_01_2019_05_01_2019.xlsx
- GiaoDichNN_ALL_04_01_2021_05_01_2021.xlsx
- GiaoDichNN_ALL_05_01_2019_06_01_2019.xlsx
- GiaoDichNN_ALL_05_01_2021_06_01_2021.xlsx
- GiaoDichNN_ALL_06_01_2020_07_01_2020.xlsx
- GiaoDichNN_ALL_06_01_2021_07_01_2021.xlsx
- GiaoDichNN_ALL_07_01_2020_08_01_2020.xlsx
- GiaoDichNN_ALL_07_01_2021_08_01_2021.xlsx
- GiaoDichNN_ALL_08_01_2019_09_01_2019.xlsx
- GiaoDichNN_ALL_09_01_2019_10_01_2019.xlsx
- GiaoDichNN_ALL_10_01_2019_11_01_2019.xlsx
- GiaoDichNN_ALL_10_01_2020_11_01_2020.xlsx
- GiaoDichNN_ALL_11_01_2020_12_01_2020.xlsx
- GiaoDichNN_ALL_12_01_2020_01_01_202

### GỘP FILE 

In [15]:
# Đọc thử file sample để xem cấu trúc dữ liệu
sample_file = ROOT_DIR / "data" / "raw" / "VN30" / "GiaoDichNN_ALL_01_01_2013_02_01_2013.xlsx"
if sample_file.exists():
    df_sample = pd.read_excel(sample_file)
    display(df_sample.head())
else:
    print(f"Không tìm thấy file: {sample_file}")

# Hàm Gộp các file
def merge_excel_files(folder_path, output_filename="merged_data.xlsx"):
    files = [f for f in os.listdir(folder_path) if f.endswith('.xlsx')]
    
    # Hàm extract ngày tháng năm từ tên file để đọc theo thứ tự cho an toàn
    def extract_start_date(filename):
        # Match pattern: GiaoDichNN_ALL_01_01_2013_02_01_2013.xlsx
        match = re.search(r"_(\d{1,2})_(\d{1,2})_(\d{4})_", filename)
        if match:
            day, month, year = int(match.group(1)), int(match.group(2)), int(match.group(3))
            return datetime(year, month, day)
        return datetime.min # Nếu không match thì xếp lên đầu

    # Sort files theo ngày bắt đầu của file
    files.sort(key=extract_start_date)
    
    print(f"Đã tìm thấy và sắp xếp {len(files)} files. Đang tiến hành gộp...")
    
    all_dfs = []
    for file in files:
        file_path = os.path.join(folder_path, file)
        try:
            df = pd.read_excel(file_path)
            all_dfs.append(df)
            print(f"  - Đã đọc {file} ({len(df)} dòng)")
        except Exception as e:
            print(f"Lỗi khi đọc file {file}: {e}")
            
    if all_dfs:
        # Nối tất cả các bảng với nhau
        merged_df = pd.concat(all_dfs, ignore_index=True)
        
        # Xoá các dòng trùng lặp (nếu có)
        merged_df = merged_df.drop_duplicates()
        
        # Format lại dữ liệu: Sắp xếp theo Mã trước, sau đó sắp xếp theo Ngày
        if 'Mã' in merged_df.columns and 'Ngày' in merged_df.columns:
            print("\nĐang định dạng (sort) lại dữ liệu theo Mã cổ phiếu và Ngày...")
            # Chuyển đổi cột Ngày sang dạng datetime chuẩn để sort (giả sử ngày đang có format dd/mm/yyyy)
            merged_df['Ngày_dt'] = pd.to_datetime(merged_df['Ngày'], format='%d/%m/%Y', errors='coerce')
            
            # Thực hiện sort: Mã (A-Z) -> Ngày (Cũ - Mới)
            merged_df = merged_df.sort_values(by=['Mã', 'Ngày_dt'], ascending=[True, True])
            
            # Xoá cột phụ Ngày_dt tạo ra khi chuẩn hoá
            merged_df = merged_df.drop(columns=['Ngày_dt'])
            
            # Reset lại index
            merged_df = merged_df.reset_index(drop=True)
        
        output_path = os.path.join(folder_path, output_filename)
        merged_df.to_excel(output_path, index=False)
        print(f"=> Đã gộp thành công {len(all_dfs)} files thành file: {output_path} (Tổng: {len(merged_df)} dòng)")
        return merged_df
    else:
        print("Không có data để gộp!")
        return None

# Chạy thử code gộp thư mục VN30
# vn30_folder = ROOT_DIR / "data" / "raw" / "VN30"
# merged_data = merge_excel_files(vn30_folder, "VN30_Merged_2013_2026.xlsx")

,Mã,Ngày,Thay đổi,KLGD ròng,GTGD ròng,KL Mua,GT Mua,KL Bán,GT Bán,Room còn lại,Đang sở hữu
0,ACB,01/02/2013,"17,70 (0,00%)",0,0.00,0,0.00,0,0.00,10,30%
1,CTG,01/02/2013,"23,40 (+1,74%)",500700,11.60,513680,11.90,12980,0.30,8784259,"12,46%"
2,FPT,01/02/2013,"39,50 (-1,50%)",1250,0.05,364160,15.62,362910,15.57,7,49%
3,GAS,01/02/2013,"45,00 (+0,67%)",37980,1.69,54780,2.43,16800,0.75,898006020,"1,61%"
4,HPG,01/02/2013,"24,70 (-0,80%)",278790,6.86,523690,12.86,244900,5.99,39051982,"39,68%"


In [34]:
vn30_folder = ROOT_DIR / "data" / "raw" / "VN30"
merged_data = merge_excel_files(vn30_folder, "VN30_Merged_2013_2026.xlsx")

Đã tìm thấy và sắp xếp 156 files. Đang tiến hành gộp...
  - Đã đọc GiaoDichNN_ALL_01_01_2013_02_01_2013.xlsx (299 dòng)
  - Đã đọc GiaoDichNN_ALL_02_01_2013_03_01_2013.xlsx (208 dòng)
  - Đã đọc GiaoDichNN_ALL_03_01_2013_04_01_2013.xlsx (286 dòng)
  - Đã đọc GiaoDichNN_ALL_04_01_2013_05_01_2013.xlsx (247 dòng)
  - Đã đọc GiaoDichNN_ALL_05_01_2013_06_01_2013.xlsx (286 dòng)
  - Đã đọc GiaoDichNN_ALL_06_01_2013_07_01_2013.xlsx (273 dòng)
  - Đã đọc GiaoDichNN_ALL_07_01_2013_08_01_2013.xlsx (312 dòng)
  - Đã đọc GiaoDichNN_ALL_08_01_2013_09_01_2013.xlsx (286 dòng)
  - Đã đọc GiaoDichNN_ALL_09_01_2013_10_01_2013.xlsx (273 dòng)
  - Đã đọc GiaoDichNN_ALL_10_01_2013_11_01_2013.xlsx (312 dòng)
  - Đã đọc GiaoDichNN_ALL_11_01_2013_12_01_2013.xlsx (273 dòng)
  - Đã đọc GiaoDichNN_ALL_12_01_2013_01_01_2014.xlsx (286 dòng)
  - Đã đọc GiaoDichNN_ALL_01_01_2014_02_01_2014.xlsx (236 dòng)
  - Đã đọc GiaoDichNN_ALL_02_01_2014_03_01_2014.xlsx (238 dòng)
  - Đã đọc GiaoDichNN_ALL_03_01_2014_04_01_2014.

In [ ]:
vnindex_folder = ROOT_DIR / "data" / "raw" / "VNINDEX"
merged_data = merge_excel_files(vnindex_folder, "VNINDEX_Merged_2013_2026.xlsx") 

Đã tìm thấy và sắp xếp 156 files. Đang tiến hành gộp...
  - Đã đọc GiaoDichNN_VNINDEX_01_1_2013_02_01_2013.xlsx (23 dòng)
  - Đã đọc GiaoDichNN_VNINDEX_02_1_2013_03_01_2013.xlsx (16 dòng)
  - Đã đọc GiaoDichNN_VNINDEX_03_1_2013_04_01_2013.xlsx (22 dòng)
  - Đã đọc GiaoDichNN_VNINDEX_04_1_2013_05_01_2013.xlsx (19 dòng)
  - Đã đọc GiaoDichNN_VNINDEX_05_1_2013_06_01_2013.xlsx (22 dòng)
  - Đã đọc GiaoDichNN_VNINDEX_06_1_2013_07_01_2013.xlsx (21 dòng)
  - Đã đọc GiaoDichNN_VNINDEX_07_1_2013_08_01_2013.xlsx (24 dòng)
  - Đã đọc GiaoDichNN_VNINDEX_08_1_2013_09_01_2013.xlsx (22 dòng)
  - Đã đọc GiaoDichNN_VNINDEX_09_1_2013_10_01_2013.xlsx (21 dòng)
  - Đã đọc GiaoDichNN_VNINDEX_10_1_2013_11_01_2013.xlsx (24 dòng)
  - Đã đọc GiaoDichNN_VNINDEX_11_1_2013_12_01_2013.xlsx (21 dòng)
  - Đã đọc GiaoDichNN_VNINDEX_12_1_2013_01_01_2014.xlsx (22 dòng)
  - Đã đọc GiaoDichNN_VNINDEX_01_1_2014_02_01_2014.xlsx (18 dòng)
  - Đã đọc GiaoDichNN_VNINDEX_02_1_2014_03_01_2014.xlsx (17 dòng)
  - Đã đọc GiaoDichN

In [ ]:
import pandas as pd
from pathlib import Path
from pathlib import Path
# Trỏ đúng về thư mục gốc của project
ROOT_DIR = find_project_root(Path.cwd())

# File Excel thực tế đang nằm ở d:\fm-forecasting\data\VNINDEX_Merged_2013_2026.xlsx
excel_path = ROOT_DIR / "data" / "VNINDEX_Merged_2013_2026.xlsx"
csv_path = ROOT_DIR / "data" / "VNINDEX_Merged_2013_2026.csv"

df = pd.read_excel(excel_path)
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("Đã lưu:", csv_path)

Đã lưu: D:\fm-forecasting\data\VNINDEX_Merged_2013_2026.csv
